# Tutorial — `arduino_elliptec`: lâminas do Arduino no lugar da Thorlabs Elliptec

O módulo [arduino_elliptec.py](../library/arduino_elliptec.py) imita a interface da Thorlabs Elliptec (o `EasyMotor` do [EASYMOTOR.py](../bell-measurements-main/EASYMOTOR.py) e o `ElliptecMotor` do pylablib). Com ele, os códigos do `bell-measurements-main` passam a controlar as lâminas do Arduino **trocando só a linha do `import`** (e, às vezes, o construtor).

**Ideia central:** na Elliptec, cada montagem tem um **endereço** (`addr`) no mesmo cabo. Aqui, cada endereço vira um **eixo** do CNC Shield (X, Y, Z, A) na mesma porta USB.

```
Thorlabs:  PC ──USB── barramento Elliptec ──┬── addr 0 (Bob)
                                            └── addr 2 (Alice)

Arduino:   PC ──USB── Arduino + CNC Shield ─┬── eixo X  = addr 0 (Bob)
                                            └── eixo Z  = addr 2 (Alice)
```

Para os detalhes de uma lâmina sozinha (perfil de velocidade, driver, fim de curso), veja o [waveplate_tutorial.ipynb](waveplate_tutorial.ipynb).

**Unidades:** ângulos em graus da lâmina, velocidades em °/s.

**Antes de começar:**
1. O firmware atualizado precisa estar gravado no Arduino.
2. Feche o `waveplate_cli.py`, o Monitor Serial e outros notebooks que usem a porta: só um programa por vez.
3. Rode as células **em ordem**, de cima para baixo. Todas as células a partir da seção 3 mexem nas lâminas.

> **Parar no meio de um movimento:** botão ■ *Interrupt* do Jupyter. As lâminas daquele comando param com desaceleração antes do erro aparecer.

## 1. Instalação

No terminal, na pasta do projeto, rode uma vez:

```bash
uv sync
```

Isso cria o `.venv` com tudo instalado, inclusive a própria biblioteca (pasta `library/`) em modo editável: `import waveplate` e `import arduino_elliptec` funcionam de **qualquer pasta** (inclusive de dentro do `bell-measurements-main`). Edições em `library/` valem na hora — só reinicie o kernel.

No VS Code, escolha o kernel **.venv** no canto superior direito do notebook. A célula abaixo confere se o kernel certo está em uso.

In [ ]:
import sys
print(sys.executable)   # deve terminar em lamina_motorizada\.venv\Scripts\python.exe

## 2. Achar a porta

O Arduino aparece como `USB-SERIAL CH340` (ou parecido). A porta dele pode ser diferente da que a Elliptec usava.

In [1]:
from serial.tools import list_ports

for p in list_ports.comports():
    print(p.device, "-", p.description)

COM4 - USB-SERIAL CH340 (COM4)


## 3. Conectar

Na Thorlabs era `stage = EasyMotor(PORT)`. Aqui há duas formas:

| Forma | Quando usar |
|---|---|
| `EasyMotor(PORT)` | mapa padrão **0→X, 1→Y, 2→Z, 3→A** (só os eixos que o firmware tiver) |
| `EasyMotor(PORT, axes={0: "X", 2: "Y"})` | quando as lâminas estão ligadas em outros eixos |

No laboratório, **Bob é o endereço 0** e **Alice é o 2**. Com o mapa padrão, Bob fica no X e Alice no Z. Ajuste `AXES` abaixo conforme os cabos estão ligados no CNC Shield.

Outras opções do construtor:

| Parâmetro | Padrão | O que faz |
|---|---|---|
| `default_addr` | o menor endereço | endereço usado quando `addr` não é passado |
| `move_timeout` | `120` | segundos máximos por movimento ou `home()` |
| `disable_when_idle` | `None` | `False`: motor sempre segurando · `True`: solta quando parado · `None`: padrão do firmware (segurando) |

Abrir a porta **reinicia o Arduino** — a conexão espera uns 2 s.

In [2]:
import time
import numpy as np

from arduino_elliptec import EasyMotor, WavePlateError

PORT = "COM4"
mountA = 2          # Alice
mountB = 0          # Bob

# endereço Elliptec → eixo do CNC Shield (ajuste conforme a ligação)
AXES = {mountB: "X", mountA: "Y"}

stage = EasyMotor(PORT, axes=AXES)
disable_when_idle = True
print(stage)

ArduinoElliptec('COM4', axes={0: 'X', 2: 'Y'})


> `EasyMotor`, `ElliptecMotor` e `ArduinoElliptec` são **o mesmo objeto** com nomes diferentes: use o nome que o código antigo já usa.

## 4. Endereços

Todo comando aceita `addr` nas formas abaixo:

| `addr` | Significado |
|---|---|
| `2` | um endereço (0 a 15) |
| `"2"`, `"F"` | endereço em hexadecimal, como na Elliptec (`"F"` = 15) |
| `[0, 2]` | vários endereços: o comando vale para todos e retorna uma **lista** |
| `"all"` | todos os endereços configurados |
| `None` (omitido) | o endereço padrão |

> **Cuidado:** `addr="A"` é o **endereço 10** (hexadecimal), não o eixo A do CNC Shield. Para o eixo A, mapeie um endereço para ele em `axes`, ex.: `axes={3: "A"}`.

In [8]:
print("Endereços configurados:", stage.get_connected_addrs())
print("Eixo de cada endereço :", {a: stage.plate(a).axis for a in stage.get_connected_addrs()})
print("Endereço padrão       :", stage.get_default_addr())

Endereços configurados: [0, 2]
Eixo de cada endereço : {0: 'X', 2: 'Y'}
Endereço padrão       : 0


`set_default_addr()` muda o endereço padrão — útil em códigos com uma lâmina só, para não repetir `addr=` em toda linha:

In [2]:
stage.set_default_addr(mountA)
print("Posição de Alice (sem addr):", stage.get_position())
print('Posição de Alice (addr="2"):', stage.get_position(addr="2"))

stage.set_default_addr(mountB)      # volta ao padrão original (o menor endereço)

Posição de Alice (sem addr): 0.0
Posição de Alice (addr="2"): 0.0


## 5. Zero mecânico: `home()`

Igual à Thorlabs: `stage.home(addr=[mountB, mountA])`. Diferenças:

- os endereços da lista fazem a referência **ao mesmo tempo**;
- demora mais: a lâmina gira até achar o fim de curso, recua e se aproxima devagar, para o zero ser repetível;
- `home_dir="cw"` é aceito, mas ignorado — o sentido da busca vem do firmware (`sentido_ref` em `EIXOS_CFG`, no [firmware/src/main.cpp](../firmware/src/main.cpp));
- o **zero é onde o fim de curso aciona**, não o zero de fábrica da Thorlabs (veja a seção 10).

Retorna a posição final (uma lista, se `addr` for lista). `stage.home(addr="all")` referencia todos.

In [3]:
stage.home()
print(stage.get_position())
stage.move_by(120)
print(stage.get_position())
stage.set_zero()
print(stage.get_position())
stage.move_by(10)
print(stage.get_position())
stage.calibrate()
print(stage.get_position())

KeyboardInterrupt: 

In [8]:
bob_zero, alice_zero = stage.home(addr=[mountB, mountA])
print(f"Depois do home: Bob = {bob_zero:.2f}°, Alice = {alice_zero:.2f}°")

Depois do home: Bob = 0.00°, Alice = 0.00°


## 6. Mover

| Função | O que faz | Retorna |
|---|---|---|
| `move_to(angle, addr=...)` | vai até o ângulo **absoluto** e espera chegar | posição final |
| `move_by(delta, addr=...)` | anda `delta` graus a partir de onde está (**relativo**) | posição final |
| `get_position(addr=...)` | lê o ângulo atual | ângulo |

Com `addr` em lista, todos andam **juntos para o mesmo ângulo** e o retorno é uma lista na mesma ordem.

In [13]:
print("Alice → 45      :", stage.move_to(45, addr=mountA))
print("Bob   → 22.5    :", stage.move_to(22.5, addr=mountB))
print("Alice + 10      :", stage.move_by(10, addr=mountA))
time.sleep(1)
print("[Bob, Alice]    :", stage.get_position(addr=[mountB, mountA]))
print("Os dois → 90    :", stage.move_to(90, addr=[mountB, mountA]))

Alice → 45      : 45.0
Bob   → 22.5    : 22.5
Alice + 10      : 54.99
[Bob, Alice]    : [22.5, 54.99]
Os dois → 90    : [90.0, 90.0]


**Por que `Alice + 10` deu 54.99 em vez de 55?** O motor anda em passos discretos (≈0,045° por passo), então o Arduino vai para o passo mais próximo e informa a posição real. Vários `move_by()` seguidos **não acumulam** esse erro: cada um parte do último ângulo *pedido*.

## 7. Alice e Bob ao mesmo tempo, em ângulos diferentes

Os códigos do laboratório fazem:

```python
stage.move_to(angle_A, addr=mountA)   # espera Alice chegar
stage.move_to(angle_B, addr=mountB)   # só então Bob começa
```

Isso funciona igual aqui, uma lâmina depois da outra. Para ganhar tempo, dá para iniciar as duas sem esperar — com `stage.plate(addr).move_to(..., wait=False)` — e depois esperar as duas com `stage.wait_move(addr=[...])`:

In [16]:
def move_pair(angle_A, angle_B):
    """Move Alice e Bob ao mesmo tempo e espera os dois. Retorna (Alice, Bob)."""
    try:
        stage.plate(mountA).move_to(angle_A, wait=False)
        stage.plate(mountB).move_to(angle_B, wait=False)
        return tuple(stage.wait_move(addr=[mountA, mountB]))
    except BaseException:           # Interrupt ou erro: não deixa ninguém andando
        stage.stop(addr=[mountA, mountB])
        raise


stage.move_to(0, addr=[mountA, mountB])

t0 = time.perf_counter()
stage.move_to(180, addr=mountA)
stage.move_to(90, addr=mountB)
print(f"Uma depois da outra: {time.perf_counter() - t0:.2f} s")

t0 = time.perf_counter()
move_pair(0, 0)
print(f"Ao mesmo tempo     : {time.perf_counter() - t0:.2f} s")

Uma depois da outra: 1.92 s
Ao mesmo tempo     : 1.38 s


## 8. Parar e verificar movimento

| Função | O que faz |
|---|---|
| `stop(addr=...)` | para com desaceleração e retorna onde parou (`addr="all"` para todos) |
| `is_moving(addr=...)` | `True` enquanto o endereço está andando |
| `wait_move(addr=...)` | espera terminar e retorna a posição final |

Os comandos de `stage` (`move_to`, `move_by`, `home`) sempre esperam o movimento terminar. Para iniciar um movimento e continuar com o Python livre, use `stage.plate(addr).move_to(..., wait=False)`, como na seção 7.

In [17]:
stage.plate(mountA).move_to(360, wait=False)      # uma volta inteira, sem esperar
time.sleep(1.0)

print("Andando [Alice, Bob]?", stage.is_moving(addr=[mountA, mountB]))
print("Alice parou em       ", stage.stop(addr=mountA))
print("Andando [Alice, Bob]?", stage.is_moving(addr=[mountA, mountB]))

stage.move_to(0, addr=mountA)

Andando [Alice, Bob]? [True, False]
Alice parou em        241.74
Andando [Alice, Bob]? [False, False]


0.0

## 9. Velocidade, aceleração e driver: `stage.plate(addr)`

A Elliptec não tem esses ajustes, então eles não fazem parte da interface `EasyMotor`. `stage.plate(addr)` devolve o objeto `WavePlate` daquele endereço, e tudo do [waveplate_tutorial.ipynb](waveplate_tutorial.ipynb) funciona nele. Cada endereço tem a sua configuração.

In [ ]:
alice = stage.plate(mountA)
print("Alice antes :", alice.get_settings())

alice.configure(max_speed=120, acceleration=200)
print("Alice depois:", alice.get_settings())
print("Bob         :", stage.plate(mountB).get_settings())

alice.restore_defaults()

- Os ajustes **somem quando o Arduino reinicia** (toda nova conexão). Para usar sempre, chame `configure()` logo depois de criar o `stage`, ou mude os padrões no firmware.
- **Para medições, mantenha o motor sempre segurando** (padrão do firmware): a lâmina não pode se mexer durante a integração. `EasyMotor(PORT, axes=AXES, disable_when_idle=False)` garante isso.

## 10. Recalibrar os ângulos das bases (`H_alice`, `H_bob`)

Os valores nos códigos do laboratório (`H_alice = 100`, `H_bob = 146.84`, …) foram medidos com o zero de fábrica da Elliptec. Aqui o zero é o fim de curso, então **esses números não valem mais** — e mudam de novo se o sensor ou a lâmina forem remontados.

Roteiro, uma lâmina por vez, depois do `home()`:
1. monte o polarizador de referência que define H;
2. gire a lâmina e meça a potência (ou as contagens) em cada ângulo;
3. o ângulo do extremo é o novo `H`.

Troque `measure()` pela leitura do seu instrumento (power meter, contagens da Time Tagger…).

In [18]:
def measure():
    """Troque pela leitura do seu instrumento."""
    return 0.0


def scan(addr, angles, settle_time=0.3):
    """Gira a lâmina `addr` por `angles`, mede em cada ponto e retorna (ângulos reais, leituras)."""
    actual, values = [], []
    for angle in angles:
        actual.append(stage.move_to(angle, addr=addr))
        time.sleep(settle_time)          # espera a lâmina parar de vibrar
        values.append(measure())
    return np.array(actual), np.array(values)

In [19]:
# Varredura grossa: 0° a 180°, de 2 em 2
coarse_angles, coarse_values = scan(mountB, np.arange(0, 181, 2))
best = coarse_angles[np.argmax(coarse_values)]   # np.argmin se o H for um mínimo na sua montagem
print(f"Grossa: extremo em {best:.2f}°")

# Varredura fina em volta do extremo
fine_angles, fine_values = scan(mountB, np.arange(best - 3, best + 3.01, 0.25))
H_bob = fine_angles[np.argmax(fine_values)]
print(f"Novo H_bob = {H_bob:.2f}°")

Grossa: extremo em 0.00°
Novo H_bob = -3.02°


In [21]:
bob = stage.plate(mountB)

In [23]:
bob.move_to(H_bob)
bob.set_zero()

In [25]:
bob.move_to(0)

0.0

Opcional — ver a curva:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(coarse_angles, coarse_values, "o-", label="grossa")
plt.plot(fine_angles, fine_values, ".-", label="fina")
plt.axvline(H_bob, color="gray", linestyle="--", label="H_bob")
plt.xlabel("ângulo de Bob (°)")
plt.ylabel("leitura")
plt.legend()
plt.show()

Repita com `mountA` para achar o `H_alice` e coloque os novos valores no código de medida. Os outros ângulos continuam relativos ao H (`V = H + 45`, `D = H + 22.5`): numa lâmina de meia onda, girar a lâmina θ gira a polarização 2θ.

## 11. Adaptando um código do `bell-measurements-main`

Exemplo com o [Bell_violation.ipynb](../bell-measurements-main/Bell_violation.ipynb). Só o começo muda:

```python
# ANTES (Thorlabs)
from EASYMOTOR import EasyMotor
PORT = "COM4"
stage = EasyMotor(PORT)

# DEPOIS (Arduino)
from arduino_elliptec import EasyMotor
PORT = "COM4"                                               # porta do Arduino
stage = EasyMotor(PORT)                                     # Bob no X, Alice no Z
# stage = EasyMotor(PORT, axes={mountB: "X", mountA: "Y"})  # se ligados em outros eixos
```

O resto continua igual: `stage.home(addr=[mountB, mountA])`, `stage.move_to(angle_A, addr=mountA)`, `stage.close()`.

Checklist:
1. **Novos `H_alice` e `H_bob`** (seção 10).
2. **Não passe `timeout` pequeno.** O `EasyMotor` da Thorlabs usava `timeout=1`; aqui o padrão é `move_timeout` (120 s), porque uma volta longa ou um `home()` passam de 1 s. Se o código antigo passa `timeout=1` explicitamente, apague — senão dá `TimeoutError`.
3. *(opcional)* Troque os dois `move_to` seguidos por `move_pair` (seção 7) para Alice e Bob girarem juntos.

## 12. Erros

| Exceção | Quando acontece |
|---|---|
| `ValueError` | detectado no Python: endereço inválido ou sem eixo, ângulo fora dos limites, dois endereços no mesmo eixo |
| `WavePlateError` | o Arduino recusou ou falhou (ex.: `fim de curso nao libera (fio solto?)` no `home()`) |
| `TimeoutError` | o Arduino não respondeu a tempo (cabo, porta, firmware travado, `timeout` curto demais) |
| `SerialException` (… *Acesso negado*) | outro programa ou outro kernel está com a porta aberta |
| `KeyboardInterrupt` | você interrompeu — as lâminas do comando já pararam |

Nenhum dos exemplos abaixo mexe nas lâminas:

In [26]:
for bad_addr in [1, 16, "G"]:
    try:
        stage.get_position(addr=bad_addr)
    except ValueError as e:
        print(f"addr={bad_addr!r:5} → ValueError: {e}")

try:
    stage.move_to(99999, addr=mountA)
except ValueError as e:
    print("ângulo      → ValueError:", e)

try:
    EasyMotor(PORT, axes={0: "X", 2: "x"})
except ValueError as e:
    print("axes        → ValueError:", e)

addr=1     → ValueError: endereço 1 não configurado (disponíveis: 0, 2); passe axes={...} ao criar o objeto
addr=16    → ValueError: endereço fora de 0..15: 16
addr='G'   → ValueError: endereço inválido: 'G'
ângulo      → ValueError: [eixo Y] ângulo 99999° fora dos limites (-3600° a 3600°)
axes        → ValueError: dois endereços apontando para o mesmo eixo


## 13. Encerrar

`close()` para as lâminas que estiverem andando, **desliga os drivers** e libera a porta serial. Rode sempre ao terminar — senão a porta fica presa e outros programas não conseguem abri-la (se acontecer, reinicie o kernel).

Para fechar mantendo os motores segurando a posição: `stage.close(disable=False)`.

In [27]:
stage.close()
print("Porta aberta?", stage.is_opened())

Porta aberta? False


Num script `.py`, prefira `with`, que fecha sozinho mesmo se der erro:

```python
with EasyMotor("COM4", axes={0: "X", 2: "Z"}) as stage:
    stage.home(addr=[0, 2])
    stage.move_to(45, addr=2)
```

## Resumo: Thorlabs × Arduino

| | Thorlabs (`EASYMOTOR`) | Arduino (`arduino_elliptec`) |
|---|---|---|
| import | `from EASYMOTOR import EasyMotor` | `from arduino_elliptec import EasyMotor` |
| conectar | `EasyMotor(PORT)` | `EasyMotor(PORT)` ou `EasyMotor(PORT, axes={...})` |
| `home()` | rápido, zero de fábrica | busca o fim de curso (mais lento); a lista de endereços referencia junta |
| `home_dir` | sentido da busca | ignorado (vem do firmware) |
| `timeout` do `move_to` | padrão 1 s | padrão `move_timeout` = 120 s |
| `max_tries` | novas tentativas | ignorado (a serial não precisa) |
| `addr` | número | número, `"0"`–`"F"`, lista ou `"all"` |
| velocidade / aceleração | — | `stage.plate(addr).configure(...)` |
| `close()` | libera a porta | para, desliga os drivers e libera a porta |

```python
from arduino_elliptec import EasyMotor

stage = EasyMotor("COM4", axes={0: "X", 2: "Z"})   # Bob = 0, Alice = 2
stage.home(addr=[0, 2])                 # zero no fim de curso (juntos)
stage.move_to(45, addr=2)               # absoluto, espera chegar
stage.move_by(-10, addr=2)              # relativo
stage.get_position(addr=[0, 2])         # [Bob, Alice]
stage.move_to(90, addr="all")           # todos para 90°
stage.stop(addr="all")
stage.plate(2).configure(max_speed=120) # ajustes extras de um endereço
stage.close()
```